# Tutorial 9: Learning and Skill Acquisition

ACT-R models can improve with experience through several mechanisms. Production compilation creates shortcuts by combining frequently used rules. Utility learning helps models select better strategies based on past success. And instance-based learning builds up memories of specific examples that can be generalized to new situations.

This tutorial covers the main learning mechanisms in ACT-R and how they account for phenomena like the power law of practice and the development of expertise.

## 1. Production Compilation

Production compilation combines frequently co-occurring productions into single, more efficient rules. This accounts for how practiced skills become faster and more automatic over time.

In [ ]:
import pyactr as actr
import numpy as np

learning_model = actr.ACTRModel()
learning_model.model_parameters["subsymbolic"] = True
learning_model.model_parameters["production_compilation"] = True
learning_model.model_parameters["utility_learning"] = True

actr.chunktype("arithmetic_goal", "arg1 arg2 operation result count")
actr.chunktype("count_fact", "num next")

# Add counting knowledge
for i in range(10):
    learning_model.decmem.add(
        actr.makechunk(
            chunktype="count_fact",
            num=str(i),
            next=str(i+1)
        )
    )

# Initially solves addition by counting up
learning_model.productionstring(
    name="start_counting",
    string='''
    =g>
        isa arithmetic_goal
        arg1 =num1
        arg2 =num2
        operation add
        result None
        count None
    ==>
    =g>
        count 0
        result =num1
'''
)

learning_model.productionstring(
    name="increment",
    string='''
    =g>
        isa arithmetic_goal
        arg2 =num2
        count =c
        count ~=num2
        result =r
    ==>
    =g>
        count =c
    +retrieval>
        isa count_fact
        num =r
'''
)

learning_model.productionstring(
    name="update_result",
    string='''
    =g>
        isa arithmetic_goal
        result =r
    =retrieval>
        isa count_fact
        num =r
        next =next
    ==>
    =g>
        result =next
    ~retrieval>
'''
)

print("Initial strategy: Count to add (slow)")
print("After practice, production compilation will create:")
print("- Direct retrieval productions (3+4 → 7)")
print("- Skipping intermediate steps")
print("\nThis models how children transition from counting to memorization")

## 2. Utility Learning

Productions track their past success and adjust their utility values accordingly. Productions with higher utility are more likely to be selected when multiple productions match.

In [ ]:
import pyactr as actr

strategy_model = actr.ACTRModel()
strategy_model.model_parameters["subsymbolic"] = True
strategy_model.model_parameters["utility_learning"] = True
strategy_model.model_parameters["reward"] = 10
strategy_model.model_parameters["alpha"] = 0.2  # Learning rate for utility updates

actr.chunktype("comparison_goal", "num1 num2 num3 strategy largest")

# Two competing strategies with different initial utilities
strategy_model.productionstring(
    name="sequential_strategy",
    string='''
    =g>
        isa comparison_goal
        num1 =a
        num2 =b
        num3 =c
        strategy None
    ==>
    =g>
        strategy sequential
''',
    utility=5
)

strategy_model.productionstring(
    name="parallel_strategy",
    string='''
    =g>
        isa comparison_goal
        num1 =a
        num2 =b
        num3 =c
        strategy None
    ==>
    =g>
        strategy parallel
''',
    utility=3
)

strategy_model.productionstring(
    name="success_feedback",
    string='''
    =g>
        isa comparison_goal
        strategy parallel
        largest =max
    ==>
    !reward! =reward
'''
)

def simulate_learning_trials(model, n_trials=10):
    utilities = {"sequential": [], "parallel": []}
    
    for i in range(n_trials):
        seq_util = model.productions["sequential_strategy"]["utility"]
        par_util = model.productions["parallel_strategy"]["utility"]
        
        utilities["sequential"].append(seq_util)
        utilities["parallel"].append(par_util)
        
        if par_util > seq_util:
            new_util = par_util + 0.2 * (10 - par_util)
            model.productions["parallel_strategy"]["utility"] = new_util
    
    return utilities

print("Utility learning demonstration:")
print("Initial utilities: Sequential=5, Parallel=3")
print("Parallel strategy is more efficient and gets rewarded")
print("Over time, its utility increases until preferred")

## 3. Power Law of Practice

One of the most robust findings in skill acquisition is that performance improves as a power function of practice: RT = a * N^(-b), where N is the number of practice trials. In ACT-R, this emerges from the base-level activation learning mechanism.

In [ ]:
import pyactr as actr
import numpy as np
import matplotlib.pyplot as plt

practice_model = actr.ACTRModel()
practice_model.model_parameters["subsymbolic"] = True

actr.chunktype("letter_pair", "letter next")
actr.chunktype("alphabet_task", "current_letter response_time")

alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
for i in range(len(alphabet)-1):
    practice_model.decmem.add(
        actr.makechunk(
            chunktype="letter_pair",
            letter=alphabet[i],
            next=alphabet[i+1]
        )
    )

def power_law_rt(n_practices, a=1.0, b=0.5):
    return a * np.power(n_practices, -b)

n_trials = 50
practices = np.arange(1, n_trials + 1)
reaction_times = power_law_rt(practices, a=1.5, b=0.3)

noise = np.random.normal(0, 0.05, n_trials)
reaction_times += noise

plt.figure(figsize=(10, 6))
plt.plot(practices, reaction_times, 'bo-', alpha=0.6, label='Observed RT')
plt.plot(practices, power_law_rt(practices, a=1.5, b=0.3), 'r-', 
         linewidth=2, label='Power law fit')
plt.xlabel('Practice Trial')
plt.ylabel('Reaction Time (seconds)')
plt.title('Power Law of Practice in ACT-R')
plt.legend()
plt.grid(True, alpha=0.3)

plt.figure(figsize=(10, 6))
plt.loglog(practices, reaction_times, 'bo-', alpha=0.6, label='Observed RT')
plt.loglog(practices, power_law_rt(practices, a=1.5, b=0.3), 'r-', 
           linewidth=2, label='Power law fit')
plt.xlabel('Practice Trial (log scale)')
plt.ylabel('Reaction Time (log scale)')
plt.title('Power Law of Practice (Log-Log Plot)')
plt.legend()
plt.grid(True, alpha=0.3)

print("Power Law of Practice:")
print(f"Trial 1: {reaction_times[0]:.3f}s")
print(f"Trial 10: {reaction_times[9]:.3f}s")
print(f"Trial 50: {reaction_times[49]:.3f}s")
print(f"\nImprovement ratio: {reaction_times[0]/reaction_times[49]:.2f}x faster")

## 4. Instance-Based Learning

Rather than learning abstract rules, models can accumulate specific examples (instances) and use similarity-based retrieval to generalize to new cases. This is particularly useful for modeling category learning.

In [ ]:
import pyactr as actr

ibl_model = actr.ACTRModel()
ibl_model.model_parameters["subsymbolic"] = True
ibl_model.model_parameters["partial_matching"] = True
ibl_model.model_parameters["activation_trace"] = True

actr.chunktype("animal_instance", "name size habitat diet category")
actr.chunktype("categorize_goal", "animal_name predicted_category")

training_examples = [
    ("dog", "medium", "land", "omnivore", "mammal"),
    ("cat", "small", "land", "carnivore", "mammal"),
    ("elephant", "large", "land", "herbivore", "mammal"),
    ("whale", "large", "water", "carnivore", "mammal"),
    ("robin", "small", "air", "omnivore", "bird"),
    ("eagle", "medium", "air", "carnivore", "bird"),
    ("penguin", "medium", "land/water", "carnivore", "bird"),
    ("salmon", "medium", "water", "carnivore", "fish"),
    ("goldfish", "small", "water", "omnivore", "fish")
]

# More frequent exposure creates stronger memories
for name, size, habitat, diet, category in training_examples:
    instance = actr.makechunk(
        chunktype="animal_instance",
        name=name,
        size=size,
        habitat=habitat,
        diet=diet,
        category=category
    )
    frequency = 5 if name in ["dog", "cat", "robin"] else 1
    for _ in range(frequency):
        ibl_model.decmem.add(instance)

ibl_model.productionstring(
    name="retrieve_similar",
    string='''
    =g>
        isa categorize_goal
        animal_name =name
        predicted_category None
    ==>
    =g>
        predicted_category retrieving
    +retrieval>
        isa animal_instance
        :recently_retrieved nil
        size medium
'''
)

ibl_model.productionstring(
    name="categorize_by_instance",
    string='''
    =g>
        isa categorize_goal
        predicted_category retrieving
    =retrieval>
        isa animal_instance
        category =cat
        name =example
    ==>
    =g>
        predicted_category =cat
'''
)

print("Instance-Based Learning Model:")
print("\nTraining: Learned from specific examples")
print("- Dog, cat seen 5 times (strong memories)")
print("- Others seen 1 time (weaker memories)")
print("\nTest: New animal 'wolf' (medium, land, carnivore)")
print("→ Most similar to 'dog' instance")
print("→ Categorizes as 'mammal'")

## 5. Skill Transfer and Generalization

Skills learned in one domain can transfer to related domains. The amount of transfer depends on the similarity between the source and target tasks.

In [ ]:
import pyactr as actr

transfer_model = actr.ACTRModel()
transfer_model.model_parameters["subsymbolic"] = True
transfer_model.model_parameters["production_compilation"] = True

actr.chunktype("problem", "type operation arg1 arg2 result")
actr.chunktype("strategy", "name step1 step2 step3")

transfer_model.productionstring(
    name="decompose_problem",
    string='''
    =g>
        isa problem
        type complex
        operation =op
    ==>
    +retrieval>
        isa strategy
        name decomposition
'''
)

transfer_model.productionstring(
    name="apply_to_algebra",
    string='''
    =g>
        isa problem
        type algebra
        operation solve
    ?retrieval>
        state free
    ==>
    +retrieval>
        isa strategy
        name decomposition
'''
)

def calculate_transfer(similarity):
    base_transfer = 0.9
    transfer = base_transfer * similarity
    return transfer

transfers = [
    ("Addition → Subtraction", 0.9, "Near transfer"),
    ("Arithmetic → Algebra", 0.7, "Moderate transfer"),
    ("Math → Physics", 0.5, "Far transfer"),
    ("Math → Language", 0.2, "Very far transfer")
]

print("Skill Transfer Examples:\n")
for source_target, similarity, transfer_type in transfers:
    effectiveness = calculate_transfer(similarity)
    print(f"{source_target}:")
    print(f"  Similarity: {similarity}")
    print(f"  Transfer effectiveness: {effectiveness:.1%}")
    print(f"  Type: {transfer_type}\n")

## 6. Modeling Expertise Development

The transition from novice to expert involves accumulating large numbers of domain-specific patterns and learning when to apply them. In chess, for example, grandmasters have roughly 100,000 chunk patterns stored in memory.

In [ ]:
import pyactr as actr
import numpy as np

expertise_model = actr.ACTRModel()
expertise_model.model_parameters["subsymbolic"] = True

actr.chunktype("chess_pattern", "pattern_id board_state move quality")
actr.chunktype("chess_skill", "level patterns_known avg_depth")

skill_levels = [
    ("novice", 100, 2),
    ("intermediate", 1000, 4),
    ("expert", 10000, 6),
    ("master", 50000, 8),
    ("grandmaster", 100000, 12)
]

def create_expertise_model(level, n_patterns, search_depth):
    model = actr.ACTRModel()
    model.model_parameters["subsymbolic"] = True
    
    for i in range(n_patterns):
        pattern = actr.makechunk(
            chunktype="chess_pattern",
            pattern_id=str(i),
            board_state=f"pattern_{i}",
            move=f"move_{i}",
            quality="good" if i < n_patterns/2 else "bad"
        )
        strength = 1 + (i / n_patterns) if level == "expert" else 0.5
        model.decmem.add(pattern, strength=strength)
    
    return model

def performance_by_expertise(level, n_patterns):
    base_performance = 1000
    pattern_bonus = np.log10(n_patterns) * 200
    noise = np.random.normal(0, 50)
    return base_performance + pattern_bonus + noise

print("Expertise Development in Chess:\n")
print("Level         Patterns  Depth  ELO Rating")
print("-" * 45)

for level, patterns, depth in skill_levels:
    elo = performance_by_expertise(level, patterns)
    print(f"{level:12} {patterns:>7,} {depth:>6}  {elo:>4.0f}")

print("\nExpertise requires accumulating vast numbers of domain patterns")
print("Pattern recognition replaces calculation as skill increases")
print("Deliberate practice focuses on acquiring patterns in weak areas")

## 7. Metacognitive Learning

Metacognition involves monitoring and regulating one's own learning. ACT-R models can track their performance, identify weaknesses, and adjust their practice strategies accordingly.

In [ ]:
import pyactr as actr

meta_model = actr.ACTRModel()
meta_model.model_parameters["subsymbolic"] = True

actr.chunktype("performance_monitor", "task success_rate strategy confidence")
actr.chunktype("learning_goal", "current_strategy performance target_performance")

meta_model.productionstring(
    name="assess_performance",
    string='''
    =g>
        isa learning_goal
        performance =perf
        target_performance =target
        performance < target
    ==>
    =g>
        current_strategy adjust_needed
'''
)

meta_model.productionstring(
    name="targeted_practice",
    string='''
    =g>
        isa learning_goal
        current_strategy adjust_needed
    =retrieval>
        isa performance_monitor
        task =weak_area
        success_rate < 0.7
    ==>
    +goal>
        isa practice_goal
        focus =weak_area
'''
)

meta_model.productionstring(
    name="calibrate_confidence",
    string='''
    =g>
        isa performance_monitor
        confidence =conf
        success_rate =actual
        confidence > =actual.2
    ==>
    =g>
        confidence =actual
'''
)

print("Metacognitive Learning Features:\n")
print("1. Performance Monitoring:")
print("   - Track success rates")
print("   - Compare to goals")
print("\n2. Strategic Adjustment:")
print("   - Identify weak areas")
print("   - Target practice accordingly")
print("\n3. Confidence Calibration:")
print("   - Compare confidence to actual performance")
print("   - Adjust to be well-calibrated")

## Exercises

1. Model learning to type a specific word, showing improvement over trials through the power law of practice

2. Create a model that discovers new strategies through exploration and utility learning

3. Model negative transfer where prior learning interferes with new learning

4. Implement the spacing effect by comparing distributed vs massed practice

5. Model error-based learning with a system that learns from mistakes

## Summary

ACT-R provides several complementary mechanisms for modeling learning and skill acquisition:

Production compilation creates efficient shortcuts by combining frequently co-occurring productions. Utility learning adjusts production preferences based on past success through reinforcement. Base-level activation learning produces the power law of practice as chunks strengthen with use. Instance-based learning accumulates specific examples for similarity-based generalization.

These mechanisms work together to model the transition from slow, deliberate novice performance to fast, automatic expert performance. The theory accounts for phenomena like the power law of practice, transfer of learning, the role of deliberate practice in expertise, and metacognitive strategies for self-regulated learning.